# 01 — Validación y preparación de datos

Este notebook verifica la estructura, calidad y coherencia de las dos bases. Conserva los archivos originales y genera versiones preparadas para el análisis descriptivo y el modelado.

In [2]:
from pathlib import Path
import unicodedata
import numpy as np
import pandas as pd
from IPython.display import display

RAIZ = Path.cwd().resolve()
if RAIZ.name == 'notebooks':
    RAIZ = RAIZ.parent
RAW = RAIZ / 'data' / 'raw'
PROCESSED = RAIZ / 'data' / 'processed'
PROCESSED.mkdir(parents=True, exist_ok=True)
pd.set_option('display.max_columns', 50)

## 1. Carga y estandarización de nombres

Los nombres se convierten internamente a etiquetas consistentes. Esto protege el análisis frente a diferencias de tildes o codificación entre archivos, sin alterar los Excel originales.

In [3]:
def clave(nombre):
    texto = unicodedata.normalize('NFKD', str(nombre)).encode('ascii', 'ignore').decode('ascii')
    return ' '.join(texto.lower().replace('_', ' ').split())

def estandarizar_columnas(df):
    columnas = {}
    for col in df.columns:
        k = clave(col)
        if k == 'id': nuevo = 'id'
        elif 'nacimiento' in k: nuevo = 'fecha_nacimiento'
        elif 'tipo cliente' in k: nuevo = 'tipo_cliente'
        elif 'factura online' in k: nuevo = 'factura_online'
        elif 'equipo' in k: nuevo = 'antiguedad_equipo'
        elif 'plan' in k and 'dato' in k: nuevo = 'plan_datos'
        elif k.startswith('factur'): nuevo = 'facturacion'
        elif k == 'mora': nuevo = 'mora'
        elif 'inicio contrato' in k: nuevo = 'fecha_inicio_contrato'
        elif k == 'minutos': nuevo = 'minutos'
        elif k == 'resultado': nuevo = 'resultado'
        else: nuevo = k.replace(' ', '_')
        columnas[col] = nuevo
    return df.rename(columns=columnas)

train = estandarizar_columnas(pd.read_excel(RAW / 'traintelco.xlsx'))
test = estandarizar_columnas(pd.read_excel(RAW / 'testelco.xlsx'))

display(pd.DataFrame({'base': ['train', 'test'], 'filas': [len(train), len(test)], 'columnas': [train.shape[1], test.shape[1]]}))

,base,filas,columnas
0,train,8243,11
1,test,2000,10


## 2. Controles de consistencia

La única columna exclusiva de entrenamiento debe ser la variable objetivo. También se revisan identificadores repetidos, tipos y faltantes.

In [4]:
assert 'resultado' in train and 'resultado' not in test
assert 'id' in train and 'id' in test

estructura = pd.DataFrame({
    'tipo_train': train.dtypes.astype(str),
    'faltantes_train': train.isna().sum(),
    'faltantes_test': test.reindex(columns=train.columns.drop('resultado')).isna().sum(),
    'valores_unicos_train': train.nunique(dropna=False)
})
display(estructura)

display(pd.DataFrame({
    'indicador': ['columnas exclusivas de train', 'columnas exclusivas de test', 'ids repetidos en train', 'ids repetidos en test'],
    'valor': [', '.join(sorted(set(train) - set(test))), ', '.join(sorted(set(test) - set(train))) or 'ninguna', train.id.duplicated().sum(), test.id.duplicated().sum()]
}))

,tipo_train,faltantes_train,faltantes_test,valores_unicos_train
antiguedad_equipo,int64,0,0.0,48
factura_online,int64,0,0.0,2
facturacion,int64,0,0.0,8151
fecha_inicio_contrato,datetime64[us],0,0.0,8243
fecha_nacimiento,datetime64[us],0,0.0,8243
id,int64,0,0.0,8243
minutos,int64,0,0.0,2215
mora,int64,0,0.0,106
plan_datos,int64,0,0.0,2
resultado,int64,0,NaN,2


,indicador,valor
0,columnas exclusivas de train,resultado
1,columnas exclusivas de test,ninguna
2,ids repetidos en train,0
3,ids repetidos en test,0


## 3. Variables temporales y controles de plausibilidad

La fecha de referencia es el 1 de enero de 2019. Se derivan edad y antigüedad contractual; los valores fuera de rangos plausibles se convierten a faltantes y quedan marcados con indicadores.

In [5]:
FECHA_CORTE = pd.Timestamp('2019-01-01')

def preparar(df):
    salida = df.copy()
    for col in ['fecha_nacimiento', 'fecha_inicio_contrato']:
        salida[col] = pd.to_datetime(salida[col], errors='coerce').dt.normalize()
    salida['edad_anios'] = (FECHA_CORTE - salida.fecha_nacimiento).dt.days / 365.25
    salida['antiguedad_contrato_meses'] = (FECHA_CORTE - salida.fecha_inicio_contrato).dt.days / 30.4375
    salida['edad_invalida'] = (~salida.edad_anios.between(18, 110)).astype('int8')
    salida['antiguedad_contrato_invalida'] = (salida.antiguedad_contrato_meses < 0).astype('int8')
    salida.loc[salida.edad_invalida.eq(1), 'edad_anios'] = np.nan
    salida.loc[salida.antiguedad_contrato_invalida.eq(1), 'antiguedad_contrato_meses'] = np.nan
    return salida

train_p = preparar(train)
test_p = preparar(test)
display(train_p[['edad_anios', 'antiguedad_contrato_meses', 'edad_invalida', 'antiguedad_contrato_invalida']].describe().T.round(2))

,count,mean,std,min,25%,50%,75%,max
edad_anios,8243.0,60.73,4.89,26.07,57.67,60.77,63.96,78.31
antiguedad_contrato_meses,8242.0,23.97,11.38,0.89,15.44,21.95,30.59,84.63
edad_invalida,8243.0,0.00,0.00,0.00,0.00,0.00,0.00,0.00
antiguedad_contrato_invalida,8243.0,0.00,0.01,0.00,0.00,0.00,0.00,1.00


## 4. Cruces clave antes del EDA

Se confirma la prevalencia de churn y se comparan las medianas numéricas entre train y test. Diferencias grandes deben discutirse como posible cambio de población, pues test no es una muestra aleatoria.

In [6]:
prevalencia = train_p.resultado.value_counts().sort_index().rename_axis('resultado').reset_index(name='clientes')
prevalencia['proporcion'] = prevalencia.clientes / prevalencia.clientes.sum()
display(prevalencia.style.format({'proporcion': '{:.1%}'}))

numericas = ['antiguedad_equipo', 'facturacion', 'mora', 'minutos', 'edad_anios', 'antiguedad_contrato_meses']
comparacion = pd.DataFrame({
    'mediana_train': train_p[numericas].median(),
    'mediana_test': test_p[numericas].median(),
    'diferencia_test_menos_train': test_p[numericas].median() - train_p[numericas].median()
}).round(2)
display(comparacion)

,resultado,clientes,proporcion
0,0,6882,83.5%
1,1,1361,16.5%


,mediana_train,mediana_test,diferencia_test_menos_train
antiguedad_equipo,24.00,26.00,2.00
facturacion,268063.00,287213.50,19150.50
mora,6.00,8.00,2.00
minutos,1103.00,1199.50,96.50
edad_anios,60.77,61.76,0.99
antiguedad_contrato_meses,21.95,22.98,1.03


## 5. Salida preparada

Las bases preparadas se guardan como CSV para que los siguientes notebooks reutilicen exactamente las mismas transformaciones. `id` se conserva para trazabilidad, pero no será un predictor.

In [7]:
# train_p.to_csv(PROCESSED / 'train_preparado.csv', index=False, encoding='utf-8')
# test_p.to_csv(PROCESSED / 'test_preparado.csv', index=False, encoding='utf-8')

# predictores = [c for c in test_p.columns if c not in ['id', 'fecha_nacimiento', 'fecha_inicio_contrato']]
# display(pd.DataFrame({'elemento': ['predictores candidatos', 'archivo train preparado', 'archivo test preparado'], 'valor': [len(predictores), 'data/processed/train_preparado.csv', 'data/processed/test_preparado.csv']}))